In [ ]:
import pandas as pd

# Step 1. Read the SUS/ICICT CSV with explicit UTF-8 encoding
csv_path = r"\ETL-Dataiku-DSS\cid10_tabela_capitulos.csv"
df = pd.read_csv(csv_path, sep=";", encoding="utf-8")
print(f"Read complete: {df.shape}")

# Step 2. Build Roman numeral → chapter code mapping
roman_to_num = {
    "I": "01","II":"02","III":"03","IV":"04","V":"05","VI":"06",
    "VII":"07","VIII":"08","IX":"09","X":"10","XI":"11","XII":"12",
    "XIII":"13","XIV":"14","XV":"15","XVI":"16","XVII":"17","XVIII":"18",
    "XIX":"19","XX":"20","XXI":"21","XXII":"22"
}

# Step 3. Read WHO official chapter labels directly from file
who_chapters = {}
with open(r"\ICD10\icd102019syst_chapters.txt", encoding="utf-8") as f:
    for line in f:
        num, label = line.strip().split(";", 1)
        who_chapters[num] = label

# Step 4. Extract Roman numeral from descricao_breve and map to WHO labels
df["icd10_chapter_code"] = df["descricao_breve"].str.split(".").str[0].str.strip()
df["icd10_chapter_code"] = df["icd10_chapter_code"].map(roman_to_num)
df["icd10_chapter_label"] = df["icd10_chapter_code"].map(who_chapters)

# Step 5. Build compact lookup table
lookup = (
    df[["codigo", "icd10_chapter_code", "icd10_chapter_label"]]
    # Defensive deduplication: not needed with current file (codes are unique),
    # but kept in case future versions change structure
    .drop_duplicates(subset=["codigo"])
    .rename(columns={"codigo": "icd10_category"})
)

# Step 6. Save with explicit UTF-8 encoding
lookup.to_csv("icd10_chapter_lookup_en.csv", index=False, encoding="utf-8")
print("✅ Saved: icd10_chapter_lookup_en.csv")


Read complete: (2560, 3)
✅ Saved: icd10_chapter_lookup_en.csv


# coverage validation: the final safeguard making sure every distinct DIAG_PRINC code in hospital admissions dataset (SIH_SUS_2024_analysis_ready) actually finds a match in lookup.

In [ ]:
import pandas as pd

# Load analysis ready admissions dataset 
df_adm = pd.read_csv(r"\SIH_SUS_2024_analysis_ready.csv", usecols=["DIAG_PRINC"], encoding="utf-8")

# Load the lookup you just built
lookup = pd.read_csv(r"\icd10_chapter_lookup_en.csv", encoding="utf-8")

# Step 1. Get distinct codes from admissions
adm_codes = set(df_adm["DIAG_PRINC"].str[:3].unique())  # truncate to 3 chars

# Step 2. Get distinct codes from lookup
lookup_codes = set(lookup["icd10_category"].unique())

# Step 3. Compare coverage
missing = adm_codes - lookup_codes
extra   = lookup_codes - adm_codes

print(f"Admissions codes: {len(adm_codes)}")
print(f"Lookup codes: {len(lookup_codes)}")
print(f"Missing codes (in admissions but not lookup): {len(missing)}")
print(f"Extra codes (in lookup but not admissions): {len(extra)}")

if missing:
    print("⚠️ Missing codes:", sorted(missing)[:20], "...")  # show first 20
else:
    print("✅ All admissions codes covered by lookup")


Admissions codes: 1666
Lookup codes: 2560
Missing codes (in admissions but not lookup): 0
Extra codes (in lookup but not admissions): 894
✅ All admissions codes covered by lookup
